# 06 · Textos: noticias y filings

**Objetivo:** reunir y limpiar las dos fuentes de **texto no estructurado** del proyecto, las noticias de cada empresa y lo que las propias empresas escriben en sus informes a la SEC, y preparar con ellas el corpus de preentrenamiento. Aquí no se calcula ningún sentimiento: solo se prepara el texto para los notebooks 07 y 08.

| Sección | Fuente | Salida en `data/cache/` |
|---|---|---|
| 1 | Noticias históricas (FNSPID, 2016–2020) | `noticias_fnspid.parquet` |
| 2 | Noticias recientes (NewsAPI, último mes) | `noticias_newsapi.parquet` (se acumula en cada ejecución) |
| 3 | Riesgos y MD&A de los 10-K y 10-Q | `filings_texto.parquet` (caché por documento en `filings_texto/`) |
| 4 | Corpus de preentrenamiento (hasta junio de 2020) | `corpus_preentrenamiento.parquet` |

**Entradas:** `data/raw/fnspid/All_external.csv` (se descarga si no existe); `universo_recortado.parquet` (02); `senales.parquet` (05); `filings_sec.parquet` y `empresas_sec.parquet` (03); claves `NEWSAPI_KEY` y `SEC_EDGAR_USER_AGENT` en el `.env`.

In [1]:
import json
import re
import time
import warnings

import pandas as pd
import plotly.express as px
import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from dotenv import load_dotenv

from funciones import (
    CARPETA_CACHE,
    PALABRA_CLAVE_A_MANO,
    SEC_PAUSA_ENTRE_PETICIONES,
    SEMILLA,
    buscar_titulares_newsapi,
    menciona_empresa,
    obtener_user_agent_sec,
    palabra_clave_empresa,
)

# Algunos filings son XHTML: BeautifulSoup avisa, pero el texto se extrae bien
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

load_dotenv();

## 1. ¿Qué noticias históricas tenemos de nuestras empresas?

Las APIs gratuitas de noticias solo dan las últimas semanas. Para tener histórico se usa **FNSPID** (Dong, Fan y Peng, 2024, [arXiv 2402.06698](https://arxiv.org/abs/2402.06698)), un dataset público de noticias financieras en el que cada titular lleva el **ticker** de su empresa. Licencia CC BY-NC 4.0: uso académico citando el artículo. El fichero ocupa 5,7 GB y solo se descarga si no está ya en disco.

In [2]:
URL_FNSPID = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/All_external.csv"
RUTA_FNSPID = CARPETA_CACHE.parent / "raw" / "fnspid" / "All_external.csv"
TAMANO_FNSPID = 5_731_397_037   # bytes, comprobado con el servidor el 19 oct 2026

if not RUTA_FNSPID.exists():
    print("Descargando FNSPID (5,7 GB)...")
    RUTA_FNSPID.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(URL_FNSPID, stream=True, timeout=60) as respuesta:
        respuesta.raise_for_status()
        with open(RUTA_FNSPID, "wb") as fichero:
            for bloque in respuesta.iter_content(chunk_size=8 * 1024 * 1024):
                fichero.write(bloque)

tamano = RUTA_FNSPID.stat().st_size
if tamano != TAMANO_FNSPID:
    raise RuntimeError(f"FNSPID incompleto: {tamano} bytes en disco, se esperaban {TAMANO_FNSPID}. Borra el fichero y vuelve a ejecutar.")
print(f"FNSPID en disco: {tamano / 1e9:.2f} GB")

FNSPID en disco: 5.73 GB


Las primeras filas, tal como vienen en el fichero:

In [3]:
pd.read_csv(RUTA_FNSPID, nrows=5)

,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
0,2020-06-05 06:30:54 UTC,Stocks That Hit 52-Week Highs On Friday,A,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-06-03 06:45:20 UTC,Stocks That Hit 52-Week Highs On Wednesday,A,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-05-26 00:30:07 UTC,71 Biggest Movers From Friday,A,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-05-22 08:45:06 UTC,46 Stocks Moving In Friday's Mid-Day Session,A,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-05-22 07:38:59 UTC,B of A Securities Maintains Neutral on Agilent...,A,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,NaN,NaN,NaN,NaN,NaN,NaN


Se usan cuatro columnas: `Date`, `Article_title` (titular), `Stock_symbol` (ticker) y `Publisher` (medio). El texto completo (`Article`) está vacío en las noticias de nuestras empresas.

**Cómo se limpia:**

1. Se lee **por trozos** de 500.000 filas (13 millones en total) y se quedan las de nuestras 1.000 empresas desde octubre de 2016. FNSPID escribe `BRK.B` y nosotros `BRK-B`: se unifica antes.
2. Se quitan los **duplicados** (misma empresa y mismo titular).
3. Se marca si el titular **habla de la empresa** (`sobre_la_empresa`). FNSPID asigna a cada empresa todo lo que sale en su página de noticias, y muchos titulares son de otras empresas: *"Altria: Income And Growth…"* aparece asignado a Texas Instruments. Un titular habla de la empresa si contiene:
   - su **ticker**, en mayúsculas y suelto: *"Plug"* no cuenta como PLUG, ni *"PG&E"* como PG, ni *"U.S."* como U. Los tickers de una letra (A, T, V…) solo cuentan entre paréntesis o tras la bolsa: *"(T)"*, *"NYSE: T"*;
   - o la **palabra clave** de su nombre: la primera palabra, o las dos primeras si la primera es demasiado común (`HORMEL FOODS CORP` → `HORMEL`, `AMERICAN AIRLINES GROUP` → `AMERICAN AIRLINES`). Titular y nombre se comparan sin puntuación: *"Bio-Techne"* → `BIO TECHNE`.

   La primera palabra falla cuando es una palabra común en los titulares financieros: *"price target"* se asignaba a T. Rowe Price, y *"Enterprise AI"* a Enterprise Products. También falla cuando dos empresas comparten la primera palabra: Intuitive Surgical e Intuitive Machines. Por eso hay una **lista revisada a mano** (`PALABRA_CLAVE_A_MANO`) con las palabras clave que comparten dos empresas, las más frecuentes en los titulares y las que dieron falsos positivos en NewsAPI. Si la empresa no tiene un nombre fiable (Target, Snap, Box, Gap…), solo cuenta el ticker. Las funciones están en `funciones.py`, porque `app.py` también las usará.

Los titulares que no hablan de su empresa **no se borran**: no sirven para el sentimiento de esa empresa, pero sí como texto para preentrenar los modelos (sección 4). Los titulares limpios se guardan en caché para no volver a leer los 5,7 GB. La marca se **recalcula en cada ejecución** (tarda unos segundos), para que un cambio en la regla llegue también a la caché.

In [4]:
RUTA_NOTICIAS_FNSPID = CARPETA_CACHE / "noticias_fnspid.parquet"
universo = pd.read_parquet(CARPETA_CACHE / "universo_recortado.parquet")
universo["palabra_clave"] = [palabra_clave_empresa(ticker, nombre) for ticker, nombre in zip(universo["ticker"], universo["name"])]
claves = universo.set_index("ticker")["palabra_clave"]
print(f"Palabras clave revisadas a mano: {len(PALABRA_CLAVE_A_MANO)} de {len(universo)} empresas"
      f" · solo por ticker: {(universo['palabra_clave'].str.len() < 3).sum()}")


def filtrar_y_limpiar_fnspid() -> pd.DataFrame:
    """Titulares de FNSPID de nuestras empresas desde oct 2016, sin duplicados."""
    trozos = []
    for trozo in pd.read_csv(RUTA_FNSPID, usecols=["Date", "Article_title", "Stock_symbol", "Publisher"],
                             dtype=str, chunksize=500_000):
        trozo["ticker"] = trozo["Stock_symbol"].str.replace(".", "-", regex=False)
        trozo["fecha"] = pd.to_datetime(trozo["Date"].str[:10])
        es_nuestro = trozo["ticker"].isin(universo["ticker"]) & (trozo["fecha"] >= "2016-10-01")
        trozos.append(trozo[es_nuestro])
    filtrado = pd.concat(trozos, ignore_index=True)
    filtrado = filtrado.rename(columns={"Article_title": "titular", "Publisher": "medio"})[["fecha", "ticker", "titular", "medio"]]
    print(f"Titulares de nuestras empresas desde oct 2016: {len(filtrado):,}")

    limpio = filtrado.dropna(subset=["titular"]).drop_duplicates(subset=["ticker", "titular"])
    print(f"Duplicados eliminados: {len(filtrado) - len(limpio):,}")
    return limpio.sort_values(["ticker", "fecha"]).reset_index(drop=True)


if RUTA_NOTICIAS_FNSPID.exists():
    print("Leído de caché:", RUTA_NOTICIAS_FNSPID.name)
else:
    filtrar_y_limpiar_fnspid().to_parquet(RUTA_NOTICIAS_FNSPID, index=False)
noticias = pd.read_parquet(RUTA_NOTICIAS_FNSPID)
noticias["sobre_la_empresa"] = [menciona_empresa(titular, ticker, claves[ticker])
                                for titular, ticker in zip(noticias["titular"], noticias["ticker"])]
noticias.to_parquet(RUTA_NOTICIAS_FNSPID, index=False)
print(f"Titulares guardados: {len(noticias):,} · sobre la propia empresa: {noticias['sobre_la_empresa'].mean():.0%}")

Palabras clave revisadas a mano: 87 de 1000 empresas · solo por ticker: 29
Leído de caché: noticias_fnspid.parquet


Titulares guardados: 464,983 · sobre la propia empresa: 41%


Ejemplos de cada tipo, para comprobar a ojo que la marca tiene sentido (arriba, sobre la empresa; abajo, no):

In [5]:
pd.set_option("display.max_colwidth", 110)
display(noticias[noticias["sobre_la_empresa"]].sample(6, random_state=4)[["fecha", "ticker", "titular"]])
display(noticias[~noticias["sobre_la_empresa"]].sample(6, random_state=4)[["fecha", "ticker", "titular"]])

,fecha,ticker,titular
388453,2019-08-21,SNPS,"Synopsys EPS beats by $0.08, beats on revenue"
73539,2018-11-28,BURL,"Burlington Stores, Inc (BURL) CEO Thomas Kingsbury on Q3 2018 Results - Earnings Call Transcript"
51992,2020-02-28,BAH,Stocks Showing Improving Market Leadership: Booz Allen Hamilton Earns 83 RS Rating
231089,2017-09-13,IVZ,Invesco's (IVZ) August AUM Increases on Source Acquisition
128300,2018-05-29,DGX,Morgan Stanley Upgrades Quest Diagnostics to Overweight
253219,2018-02-07,KR,"South Texas Money Management Ltd Buys American Tower Corp, The Kroger Co, Tencent Holdings, ..."


,fecha,ticker,titular
234074,2017-03-13,JCI,8 Biggest Price Target Changes For Monday
55880,2018-03-29,BEN,"Templeton Emerging Markets Fund (""EMF"") Announces Results of the Reconvened Annual ..."
433456,2020-02-10,URI,"Construction Stock Q4 Earnings Due on Feb 11: MAS, MLM, LPX"
90278,2020-03-08,CI,"Accolade's $100 Million IPO: A Good, But Overvalued Business"
52542,2019-07-08,BAM,Screening For Critical Mission Infrastructure Value
225035,2019-12-01,INTC,Top Holdings Of Dividend ETFs (Part 1: The Top 50 In November 2019)


¿Qué empresas y qué meses cubren los titulares que sí hablan de su empresa?

In [6]:
sobre_empresa = noticias[noticias["sobre_la_empresa"]]
por_empresa = sobre_empresa.groupby("ticker").size()
print(f"Periodo: {noticias['fecha'].min().date()} → {noticias['fecha'].max().date()}")
print(f"Empresas con algún titular propio: {len(por_empresa)} de {len(universo)} · mediana por empresa: {por_empresa.median():.0f}")

por_mes = sobre_empresa.groupby(pd.Grouper(key="fecha", freq="ME")).size().reset_index(name="titulares")
grafico = px.bar(por_mes, x="fecha", y="titulares", title="Titulares sobre la propia empresa, por mes (FNSPID)")
grafico.show(renderer="notebook_connected")

Periodo: 2016-10-01 → 2020-06-11
Empresas con algún titular propio: 633 de 1000 · mediana por empresa: 263


**Conclusión:** FNSPID da noticias de unas dos terceras partes de nuestras empresas (el 69 % de las de NYSE y el 54 % de las de Nasdaq) entre octubre de 2016 y junio de 2020. Menos de la mitad de los titulares hablan de su propia empresa; esos medirán su sentimiento y el resto se aprovecha como corpus. De las ~370 empresas sin noticias, 119 empezaron a cotizar después de 2020 y el resto no está en FNSPID; quedan fuera de la comparación noticias/filings.

## 2. ¿Qué se dice ahora de las empresas con señal?

FNSPID termina en junio de 2020. Para el screener del día y para el test que se etiqueta a mano hacen falta noticias **recientes**. Se usan las de **NewsAPI**, con los límites del plan gratuito medidos el 19 oct 2026:

| Límite | Valor |
|---|---|
| Peticiones | 100 al día |
| Antigüedad máxima | un mes |
| Resultados por consulta | 100 como máximo |

Por eso se consultan solo las empresas con **señal el último día** de nuestros precios, en grupos de 3 por consulta. Se busca solo **en el titular** y solo en **26 medios financieros**: sin ese filtro, buscar "Garmin" devuelve sorteos de relojes. Así llega un artículo real (respuesta guardada en caché para no gastar cupo):

In [7]:
RUTA_PRUEBA_NEWSAPI = CARPETA_CACHE / "newsapi_respuesta_ejemplo.json"
if not RUTA_PRUEBA_NEWSAPI.exists():
    articulos_garmin = buscar_titulares_newsapi(["Garmin"], desde=pd.Timestamp.today() - pd.Timedelta(days=29))
    RUTA_PRUEBA_NEWSAPI.write_text(json.dumps(articulos_garmin, indent=1))
json.loads(RUTA_PRUEBA_NEWSAPI.read_text())[0]

{'source': {'id': None, 'name': 'PRNewswire'},
 'author': None,
 'title': 'Garmin selects Ventura, California as newest Garmin Marathon Series location',
 'description': 'Event will showcase breathtaking mountain and ocean scenery in August 2027 OLATHE, Kan., Oct. 14, 2026 /PRNewswire/ -- Garmin (NYSE: GRMN) today announced the Garmin Marathon Series will expand to a fourth location in Ventura, California, on Sunday, Aug. 22, …',
 'url': 'https://www.prnewswire.com/news-releases/garmin-selects-ventura-california-as-newest-garmin-marathon-series-location-302908287.html',
 'urlToImage': 'https://mmx.prnewswire.com/media/MS1128164/Garmin-Marathon-Series.jpg?id=OA3016159&p=facebook',
 'publishedAt': '2026-10-14T14:02:00Z',
 'content': 'Event will showcase breathtaking mountain and ocean scenery in August 2027\r\nOLATHE, Kan., Oct. 14, 2026 /PRNewswire/ -- Garmin (NYSE: GRMN) today announced the Garmin Marathon Series will expand to a… [+5337 chars]'}

Cada titular se asigna a las empresas de su grupo que **menciona** (la misma regla que en FNSPID). Como NewsAPI solo guarda un mes, cada ejecución **se suma** a lo ya guardado sin repetir artículos. Si ya se descargó hoy, no se pide nada. Se busca por la palabra clave o, si la empresa solo cuenta por ticker, por el ticker. Las noticias guardadas en días anteriores se vuelven a filtrar con la regla actual.

In [8]:
RUTA_NOTICIAS_NEWSAPI = CARPETA_CACHE / "noticias_newsapi.parquet"
EMPRESAS_POR_CONSULTA = 3
hoy = pd.Timestamp.today().normalize()

senales = pd.read_parquet(CARPETA_CACHE / "senales.parquet", columns=["fecha", "ticker", "senal"])
ultimo_dia = senales["fecha"].max()
con_senal = senales[(senales["fecha"] == ultimo_dia) & senales["senal"].isin(["compra", "venta"])]
empresas = universo[universo["ticker"].isin(con_senal["ticker"])].copy()
empresas["termino_busqueda"] = empresas["palabra_clave"].where(empresas["palabra_clave"].str.len() >= 3, empresas["ticker"])
print(f"Empresas con señal el {ultimo_dia.date()}: {len(empresas)} → {-(-len(empresas) // EMPRESAS_POR_CONSULTA)} consultas")

Empresas con señal el 2026-10-09: 109 → 37 consultas


In [9]:
def asignar_a_empresas(articulos: list[dict], grupo: pd.DataFrame) -> list[dict]:
    """Una fila por cada artículo y cada empresa del grupo cuyo ticker o palabra clave sale en el titular."""
    filas = []
    for articulo in articulos:
        for empresa in grupo.itertuples():
            if menciona_empresa(articulo["title"], empresa.ticker, empresa.palabra_clave):
                filas.append({"fecha": pd.Timestamp(articulo["publishedAt"][:10]), "ticker": empresa.ticker,
                              "titular": articulo["title"], "resumen": articulo["description"],
                              "medio": articulo["source"]["name"], "url": articulo["url"], "descargado": hoy})
    return filas


guardadas = pd.read_parquet(RUTA_NOTICIAS_NEWSAPI) if RUTA_NOTICIAS_NEWSAPI.exists() else None
if guardadas is not None and guardadas["descargado"].max() >= hoy:
    print("Ya se descargó hoy: no se hace ninguna petición")
else:
    filas, consultas_llenas = [], 0
    for inicio in range(0, len(empresas), EMPRESAS_POR_CONSULTA):
        grupo = empresas.iloc[inicio : inicio + EMPRESAS_POR_CONSULTA]
        articulos = buscar_titulares_newsapi(list(grupo["termino_busqueda"]), desde=hoy - pd.Timedelta(days=29))
        consultas_llenas += len(articulos) == 100
        filas += asignar_a_empresas(articulos, grupo)
    print(f"Artículos nuevos asignados: {len(filas)} · consultas que llegaron al tope de 100: {consultas_llenas}")
    nuevas = pd.DataFrame(filas) if guardadas is None else pd.concat([guardadas, pd.DataFrame(filas)], ignore_index=True)
    nuevas.drop_duplicates(subset=["ticker", "url"]).to_parquet(RUTA_NOTICIAS_NEWSAPI, index=False)

noticias_recientes = pd.read_parquet(RUTA_NOTICIAS_NEWSAPI)
sigue_valiendo = [menciona_empresa(titular, ticker, claves[ticker])
                  for titular, ticker in zip(noticias_recientes["titular"], noticias_recientes["ticker"])]
if not all(sigue_valiendo):
    print(f"Guardadas que ya no cumplen la regla actual, eliminadas: {len(sigue_valiendo) - sum(sigue_valiendo)}")
    noticias_recientes = noticias_recientes[sigue_valiendo].reset_index(drop=True)
    noticias_recientes.to_parquet(RUTA_NOTICIAS_NEWSAPI, index=False)
print(f"Noticias recientes guardadas: {len(noticias_recientes)} · empresas: {noticias_recientes['ticker'].nunique()}"
      f" · de {noticias_recientes['fecha'].min().date()} a {noticias_recientes['fecha'].max().date()}")

Artículos nuevos asignados: 1154 · consultas que llegaron al tope de 100: 3
Noticias recientes guardadas: 1485 · empresas: 101 · de 2026-09-20 a 2026-10-22


In [10]:
display(noticias_recientes.sample(8, random_state=1)[["fecha", "ticker", "medio", "titular"]])
noticias_recientes["medio"].value_counts().head(5)

,fecha,ticker,medio,titular
201,2026-10-05,V,Business Insider,"After 11 years on an H-1B visa, I still miss my family back home. Here’s why I stay in the US."
390,2026-10-13,T,Biztoc.com,Starlink Is Getting A Key Upgrade To Start Competing With T-Mobile And AT&T
91,2026-10-12,PLTR,Biztoc.com,Palantir Stock Gets Fresh Wall Street Vote of Confidence
631,2026-10-05,MAR,Biztoc.com,"A Hilton and Marriott Hotel Operator Raised $86 Million. The SEC Says It Misled More Than 2,000 Investors"
766,2026-10-04,SU,Biztoc.com,Suncor Energy (TSX:SU) Could Be 8% Undervalued After Pathways CCS Deal
1081,2026-10-18,WMT,Biztoc.com,"Halloween Candy Price Check: Walmart, Target, Publix and more"
401,2026-10-09,T,Biztoc.com,"SpaceX snaps up the 800 MHz spectrum T-Mobile dumped for $2.9 billion — and T-Mobile, Verizon, AT&T stocks..."
325,2026-10-06,PG,Biztoc.com,"Here Are Tuesday’s Top Wall Street Analyst Research Calls: Borg Warner, Boyd Gaming, JetBlue, Lattice Semi..."


medio
Biztoc.com          1195
PRNewswire           209
Business Insider      25
CNBC                  25
24/7 Wall St.         22
Name: count, dtype: int64

**Conclusión:** NewsAPI aporta titular **y resumen** de las empresas con señal, solo del último mes, y el histórico reciente crece con cada ejecución en días distintos. Casi todo llega por Biztoc (un agregador) y PR Newswire. Limitaciones: algunas consultas llegan al tope de 100 artículos. La regla de mención se revisó el 22 oct 2026: con la primera palabra del nombre, unas 500 de 1.822 noticias (27 %) eran de otra cosa (*"Gold price today"* asignado a T. Rowe Price). Con la lista a mano quedan errores sueltos (*"…at AT&T Stadium"*), y las empresas que solo cuentan por ticker (Target, Nasdaq, Snap…) pierden los titulares que no lo escriben.

## 3. ¿Qué dicen las empresas de sí mismas? (filings de la SEC)

La segunda fuente de texto son los informes que las empresas de EEUU presentan por ley a la SEC:

| | **10-K** | **10-Q** |
|---|---|---|
| Qué es | Informe **anual**, con cuentas auditadas | Informe **trimestral** (3 al año) |
| Riesgos | **Item 1A**: todos los riesgos del negocio | **Part II, Item 1A**: solo los cambios desde el último 10-K |
| Análisis de la dirección (MD&A) | **Item 7**: la dirección explica el año | **Part I, Item 2**: lo mismo para el trimestre |

Se extraen esas dos partes, **riesgos** y **MD&A**: el texto en el que la propia empresa habla de su situación. Alcance decidido por el autor el 20 oct 2026: 10-K y 10-Q de las empresas que presentan 10-K (las ~110 extranjeras, con 20-F/40-F, quedan fuera); de **octubre de 2016 a junio de 2020** más el **último 10-K y 10-Q** de cada empresa para el screener; XOM con su CIK anterior (notebook 03).

In [11]:
filings = pd.read_parquet(CARPETA_CACHE / "filings_sec.parquet")
empresas_sec = pd.read_parquet(CARPETA_CACHE / "empresas_sec.parquet")
filings = filings.merge(empresas_sec[["cik", "ticker", "formulario_anual"]], on="cik")
filings = filings[filings["formulario"].isin(["10-K", "10-Q"]) & (filings["formulario_anual"] == "10-K")]

en_el_periodo = filings[(filings["fecha_presentacion"] >= "2016-10-01") & (filings["fecha_presentacion"] <= "2020-06-30")]
los_ultimos = filings.sort_values("fecha_presentacion").groupby(["cik", "formulario"]).tail(1)
documentos = pd.concat([en_el_periodo, los_ultimos]).drop_duplicates(subset=["cik", "numero_acceso"]).reset_index(drop=True)

print(f"Documentos a procesar: {len(documentos):,} de {documentos['cik'].nunique()} empresas")
documentos["formulario"].value_counts()

Documentos a procesar: 12,523 de 883 empresas


formulario
10-Q    8830
10-K    3693
Name: count, dtype: int64

### 3.1 Cómo es un filing por dentro

Cada filing es una página HTML de varios MB. Se descarga con el User-Agent que exige la SEC (con reintentos si responde "demasiadas peticiones"), se le quitan las etiquetas y se juntan los espacios.

In [12]:
def descargar_documento_sec(url: str, intentos: int = 3) -> bytes:
    """Descarga un documento de EDGAR con el User-Agent obligatorio, esperando entre peticiones.

    Reintenta si la SEC responde 429 o un error 5xx. Lanza RuntimeError si no lo consigue.
    """
    for intento in range(1, intentos + 1):
        try:
            respuesta = requests.get(url, headers={"User-Agent": obtener_user_agent_sec()}, timeout=60)
        except requests.exceptions.RequestException as error:
            motivo = str(error)
        else:
            time.sleep(SEC_PAUSA_ENTRE_PETICIONES)
            if respuesta.status_code == 200:
                return respuesta.content
            motivo = f"HTTP {respuesta.status_code}"
            if respuesta.status_code not in (429, 500, 502, 503, 504):
                break
        time.sleep(2 * intento)
    raise RuntimeError(f"No se pudo descargar {url}: {motivo}")


def html_a_texto(contenido: bytes) -> str:
    """Texto plano de un documento HTML, con los espacios juntados en uno."""
    texto = BeautifulSoup(contenido, "lxml").get_text(" ").replace("\xa0", " ")
    return re.sub(r"\s+", " ", texto)


def url_del_documento(filing) -> str:
    """Dirección del documento principal de un filing en EDGAR."""
    carpeta = filing.numero_acceso.replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{int(filing.cik_archivo)}/{carpeta}/{filing.documento_principal}"

Ejemplo real: el 10-K de Apple de 2019. ¿Dónde aparece la cabecera "Item 1A"?

In [13]:
RUTA_EJEMPLO = CARPETA_CACHE / "filing_ejemplo_aapl_10k_2019.txt"
filing_apple = documentos[(documentos["ticker"] == "AAPL") & (documentos["formulario"] == "10-K")
                          & (documentos["fecha_presentacion"].dt.year == 2019)].iloc[0]
if not RUTA_EJEMPLO.exists():
    RUTA_EJEMPLO.write_text(html_a_texto(descargar_documento_sec(url_del_documento(filing_apple))))
texto_apple = RUTA_EJEMPLO.read_text()

print(f"10-K de Apple ({filing_apple['fecha_presentacion'].date()}): {len(texto_apple):,} caracteres")
for aparicion in re.finditer(r"item\s*1a", texto_apple, flags=re.I):
    print(f"  posición {aparicion.start():>7}: …{texto_apple[aparicion.start() - 20 : aparicion.start() + 60]}…")

10-K de Apple (2019-10-31): 262,791 caracteres
  posición   29125: … Item 1. Business 1 Item 1A. Risk Factors 5 Item 1B. Unresolved Staff Comments 1…
  posición   31362: …iscussed in Part I, Item 1A of this Form 10-K under the heading “Risk Factors,” …
  posición   45123: … 2019 Form 10-K | 4 Item 1A. Risk Factors The following discussion of risk facto…
  posición  101630: …rights ” in Part I, Item 1A of this Form 10-K under the heading “Risk Factors.” …
  posición  108153: …iscussed in Part I, Item 1A of this Form 10-K under the heading “Risk Factors,” …
  posición  117632: …et forth in Part I, Item 1A of this Form 10-K under the heading “Risk Factors”. …


Aparece **varias veces**: en el **índice** (seguida del número de página), como **referencia** dentro del texto (*"…Item 1A of this Form 10-K under the heading “Risk Factors”"*) y como **cabecera real**, seguida de los riesgos.

### 3.2 Cómo se localiza cada sección

Cada sección tiene una cabecera de **inicio** y una lista de cabeceras de **final** por orden de preferencia: los riesgos de un 10-K acaban en el Item 1B; si no lo hay, en el 1C; si no, en el 2. Las reglas:

- Se **descartan las referencias**: cabeceras precedidas de comillas, *see*, *in*, *under*, *of* o *and*, o seguidas de comillas (también tras un punto o un guion: *Risk Factors."*, *Risk Factors - "…"*).
- De cada inicio se corta hasta el **primer final** que aparece después, y entre todos los candidatos se queda el **más largo**: el del índice mide unos pocos caracteres y el real, decenas de miles.
- Si la sección ocupa **más del 75 % del documento**, se descarta, porque el corte se ha llevado casi todo el informe. En una muestra de 80 documentos, las secciones correctas llegaban como mucho al 68 % y las erróneas al 90–95 %.

In [14]:
SEPARADOR = r"[\s\.\:\-—–]*"   # entre "Item", el número y el título puede haber puntos, guiones o dos puntos


def cabecera(numero: str, titulo: str) -> str:
    """Expresión regular de una cabecera como «Item 1A. Risk Factors» (sin distinguir mayúsculas)."""
    return rf"item{SEPARADOR}{numero}{SEPARADOR}" + titulo.replace(" ", r"\s+")


# (formulario, sección): (cabecera de inicio, cabeceras de final por orden de preferencia)
SECCIONES = {
    ("10-K", "riesgos"): (cabecera("1a", "risk factors"),
                          [cabecera("1b", "unresolved"), cabecera("1c", "cybersecurity"), cabecera("2", "(?:description of )?propert")]),
    ("10-K", "mda"): (cabecera("7", "management.{0,3}s? discussion"),
                      [cabecera("7a", "quantitative"), cabecera("8", "financial statements")]),
    ("10-Q", "riesgos"): (cabecera("1a", "risk factors"),
                          [cabecera("2", "unregistered"), cabecera("3", "defaults"), cabecera("4", "mine"),
                           cabecera("5", "other information"), cabecera("6", "exhibits")]),
    ("10-Q", "mda"): (cabecera("2", "management.{0,3}s? discussion"),
                      [cabecera("3", "quantitative"), cabecera("4", "controls")]),
}

Con esas cabeceras, la extracción aplica las reglas: descarta las citas, corta en el primer final disponible, se queda con el tramo más largo y lo descarta si ocupa más del 75 % del documento.

In [15]:
MAXIMO_DEL_DOCUMENTO = 0.75
ANTES_DE_UNA_REFERENCIA = re.compile(r"(?:[“\"‘]|\bin|\bsee|\bunder|\bof|\band)\s*$", flags=re.I)


def es_referencia(texto: str, inicio: re.Match) -> bool:
    """True si la cabecera encontrada es una cita a la sección y no la sección misma."""
    antes = texto[max(0, inicio.start() - 8) : inicio.start()]
    despues = texto[inicio.end() : inicio.end() + 5].lstrip(" .,;:-–—")   # «Risk Factors."» o «Risk Factors - "…"» también son citas
    return bool(ANTES_DE_UNA_REFERENCIA.search(antes)) or despues[:1] in ("”", '"', "’")


def extraer_seccion(texto: str, formulario: str, parte: str) -> str:
    """Texto de una sección ('riesgos' o 'mda') de un 10-K o 10-Q; cadena vacía si no se encuentra."""
    patron_inicio, patrones_final = SECCIONES[(formulario, parte)]
    mejor = ""
    for inicio in re.finditer(patron_inicio, texto, flags=re.I):
        if es_referencia(texto, inicio):
            continue
        for patron_final in patrones_final:
            final = re.compile(patron_final, flags=re.I).search(texto, inicio.end())
            if final:
                if final.start() - inicio.start() > len(mejor):
                    mejor = texto[inicio.start() : final.start()]
                break
    if len(mejor) > MAXIMO_DEL_DOCUMENTO * len(texto):
        return ""
    return mejor

In [16]:
for parte in ["riesgos", "mda"]:
    seccion = extraer_seccion(texto_apple, "10-K", parte)
    print(f"{parte}: {len(seccion):,} caracteres · empieza: «{seccion[:90]}» · acaba: «{seccion[-60:]}»")

riesgos: 54,890 caracteres · empieza: «Item 1A. Risk Factors The following discussion of risk factors contains forward-looking st» · acaba: «d operating results could be materially adversely affected. »
mda: 33,810 caracteres · empieza: «Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operati» · acaba: «at reporting period could be materially adversely affected. »


### 3.3 Descargar y extraer los ~12.500 documentos

Cada documento se guarda **en su propio fichero** de `data/cache/filings_texto/`. Si la descarga se corta, se sigue por donde iba, y re-ejecutar el notebook no vuelve a pedir nada a la SEC. La primera vez tardó **2 h 24 min** (unos 3,6 MB por documento, dentro del límite de la SEC). Un documento que falla se anota y no detiene el resto.

In [17]:
CARPETA_FILINGS_TEXTO = CARPETA_CACHE / "filings_texto"
CARPETA_FILINGS_TEXTO.mkdir(exist_ok=True)

errores_filings = {}
inicio_descarga = time.time()
pendientes = [f for f in documentos.itertuples() if not (CARPETA_FILINGS_TEXTO / f"{f.numero_acceso}.json").exists()]
print(f"Ya en caché: {len(documentos) - len(pendientes):,} · pendientes: {len(pendientes):,}")

for numero, filing in enumerate(pendientes, start=1):
    try:
        texto = html_a_texto(descargar_documento_sec(url_del_documento(filing)))
    except RuntimeError as error:
        errores_filings[filing.numero_acceso] = str(error)
        continue
    resultado = {"caracteres_documento": len(texto),
                 "riesgos": extraer_seccion(texto, filing.formulario, "riesgos"),
                 "mda": extraer_seccion(texto, filing.formulario, "mda")}
    (CARPETA_FILINGS_TEXTO / f"{filing.numero_acceso}.json").write_text(json.dumps(resultado))
    if numero % 500 == 0:
        print(f"  {numero:,}/{len(pendientes):,} ({(time.time() - inicio_descarga) / 60:.0f} min)")

print(f"Errores de descarga: {len(errores_filings)}", errores_filings if errores_filings else "")

Ya en caché: 12,523 · pendientes: 0
Errores de descarga: 0 


### 3.4 ¿Se han encontrado las secciones?

Se juntan los resultados en una tabla (una fila por filing) y se mide en cuántos documentos se ha encontrado cada sección.

In [18]:
filas = []
for filing in documentos.itertuples():
    ruta = CARPETA_FILINGS_TEXTO / f"{filing.numero_acceso}.json"
    if ruta.exists():
        filas.append({"numero_acceso": filing.numero_acceso, **json.loads(ruta.read_text())})
filings_texto = documentos[["cik", "ticker", "formulario", "fecha_presentacion", "fecha_periodo", "numero_acceso"]].merge(
    pd.DataFrame(filas), on="numero_acceso")
filings_texto["caracteres_riesgos"] = filings_texto["riesgos"].str.len()
filings_texto["caracteres_mda"] = filings_texto["mda"].str.len()
filings_texto.to_parquet(CARPETA_CACHE / "filings_texto.parquet", index=False)

filings_texto["tiene_riesgos"] = filings_texto["caracteres_riesgos"] > 0
filings_texto["tiene_mda"] = filings_texto["caracteres_mda"] > 0
resumen = filings_texto.groupby("formulario").agg(
    documentos=("numero_acceso", "size"),
    con_riesgos=("tiene_riesgos", "mean"),
    caracteres_riesgos_mediana=("caracteres_riesgos", "median"),
    con_mda=("tiene_mda", "mean"),
    caracteres_mda_mediana=("caracteres_mda", "median"),
)
resumen[["con_riesgos", "con_mda"]] = (resumen[["con_riesgos", "con_mda"]] * 100).round(0)   # en %
print(f"Filings con texto guardado: {len(filings_texto):,} de {len(documentos):,} · empresas: {filings_texto['cik'].nunique()}")
resumen

Filings con texto guardado: 12,523 de 12,523 · empresas: 883


,documentos,con_riesgos,caracteres_riesgos_mediana,con_mda,caracteres_mda_mediana
formulario,,,,,
10-K,3693,98.0,65382.0,98.0,72996.0
10-Q,8830,89.0,428.0,97.0,49023.0


In [19]:
sin_cambios = filings_texto["riesgos"].str.contains("no material change", case=False)
print(f"Riesgos de los 10-Q que dicen «no material changes»: {sin_cambios[filings_texto['formulario'] == '10-Q'].mean():.0%} · ejemplo:")
print("  ", filings_texto.loc[sin_cambios & (filings_texto["formulario"] == "10-Q"), "riesgos"].iloc[0][:170])

sin_nada = filings_texto[~filings_texto["tiene_riesgos"] & ~filings_texto["tiene_mda"]]
print(f"Documentos sin ninguna de las dos secciones: {len(sin_nada)} ({len(sin_nada) / len(filings_texto):.1%})"
      f" · los que más: {', '.join(sin_nada['ticker'].value_counts().head(3).index)}")

Riesgos de los 10-Q que dicen «no material changes»: 45% · ejemplo:
   ITEM 1A. RISK FACTORS Refer to the description of the risk factors associated with our business previously disclosed in Item 1A of our Annual Report on Form 10-K for the 
Documentos sin ninguna de las dos secciones: 158 (1.3%) · los que más: C, EIX, MOH


**Conclusión:** cuando la empresa usa las cabeceras estándar, la extracción encuentra las dos secciones casi siempre (98 % en los 10-K). En los 10-Q, casi la mitad de los apartados de riesgos (45 %) solo dicen que no hay cambios, así que su texto con contenido es sobre todo el MD&A. El 1,3 % de documentos sin secciones son sobre todo bancos y eléctricas con formato propio; Citigroup y Edison International no tienen texto en ningún filing y quedan sin sentimiento de filings.

## 4. ¿Con qué texto aprenden a leer los modelos? (corpus de preentrenamiento)

En el notebook 07, el BERT y el GPT propios se **preentrenan** leyendo mucho texto financiero **sin etiquetas**: el BERT adivina palabras tapadas y el GPT, la palabra siguiente. Ese texto es el **corpus**: todos los titulares de FNSPID (también los que no hablan de su empresa) y frases de los filings. Tres reglas:

1. **Solo textos hasta el 30 de junio de 2020**, el final del periodo con noticias.
2. **Las noticias de NewsAPI no entran:** de ellas saldrán los ~200 titulares del test etiquetado a mano, y el modelo no debe haberlos leído.
3. **Tamaño fijo de unos 15–17 millones de palabras**, frente a los ~128 M de los filings del periodo: con lo medido el 20 oct (~52.000 tokens/s con PyTorch) son unos 25 minutos por modelo.

In [20]:
FECHA_LIMITE_CORPUS = "2020-06-30"

titulares_corpus = (noticias[noticias["fecha"] <= FECHA_LIMITE_CORPUS]
                    .drop_duplicates(subset="titular")[["titular", "ticker", "fecha"]]
                    .rename(columns={"titular": "texto"}))
titulares_corpus["fuente"] = "titular"
titulares_corpus["documento"] = "fnspid"   # cada titular es un documento por sí mismo
print(f"Titulares distintos: {len(titulares_corpus):,} · palabras: {titulares_corpus['texto'].str.split().str.len().sum() / 1e6:.1f} M")

Titulares distintos: 294,536 · palabras: 3.4 M


### 4.1 De las secciones de los filings a frases

Cada sección se parte en frases (corte tras `.`, `!` o `?` seguido de espacio y mayúscula). Al pasar el HTML a texto quedan restos de tablas, así que una frase solo se queda si mide **entre 30 y 600 caracteres** y al menos el **60 % son letras**. También se quitan las **frases repetidas**, los textos legales que cada empresa copia trimestre tras trimestre: si no, el modelo los aprendería de memoria.

In [21]:
CORTE_DE_FRASE = re.compile(r"(?<=[.!?])\s+(?=[A-Z“\"(])")


def partir_en_frases(texto: str) -> list[str]:
    """Parte un texto en frases: corta tras '.', '!' o '?' seguidos de espacio y mayúscula."""
    return [frase.strip() for frase in CORTE_DE_FRASE.split(texto) if frase.strip()]


def es_frase_util(frase: str) -> bool:
    """True si la frase mide entre 30 y 600 caracteres y al menos el 60 % son letras."""
    if not 30 <= len(frase) <= 600:
        return False
    letras = sum(caracter.isalpha() for caracter in frase)
    return letras >= 0.6 * len(frase)


ejemplo = "Net sales increased 6% in 2019. See Part I, Item 1. Revenue was $260,174 $265,595 $229,234. Our results may be adversely affected by tariffs."
for frase in partir_en_frases(ejemplo):
    print("se queda " if es_frase_util(frase) else "se quita ", "|", frase)

se queda  | Net sales increased 6% in 2019.
se quita  | See Part I, Item 1.
se quita  | Revenue was $260,174 $265,595 $229,234.
se queda  | Our results may be adversely affected by tariffs.


In [22]:
filings_corpus = filings_texto[filings_texto["fecha_presentacion"] <= FECHA_LIMITE_CORPUS].sort_values("fecha_presentacion")

filas = []
for filing in filings_corpus.itertuples():
    for fuente in ["riesgos", "mda"]:
        for frase in partir_en_frases(getattr(filing, fuente)):
            if es_frase_util(frase):
                filas.append({"texto": frase, "ticker": filing.ticker, "fecha": filing.fecha_presentacion,
                              "fuente": f"filing_{fuente}", "documento": filing.numero_acceso})
frases_filings = pd.DataFrame(filas)
repetidas = frases_filings.duplicated(subset="texto")
print(f"Frases útiles de {len(filings_corpus):,} filings: {len(frases_filings):,} · repetidas: {repetidas.mean():.0%}")
frases_filings = frases_filings[~repetidas]
print(f"Sin repetir: {len(frases_filings):,} frases · {frases_filings['texto'].str.split().str.len().sum() / 1e6:.0f} M palabras")

Frases útiles de 10,758 filings: 4,297,048 · repetidas: 51%


Sin repetir: 2,097,070 frases · 70 M palabras


### 4.2 Recortar al tamaño fijado

Se toman **como mucho 40 frases al azar de cada filing**, en lugar de una muestra al azar del total, para que las empresas con informes enormes (bancos, aseguradoras) no dominen el corpus. La semilla fija hace que la muestra sea siempre la misma.

In [23]:
FRASES_POR_FILING = 40

# Se barajan todas las frases (con la semilla fija) y de cada filing se cogen las 40 primeras
barajadas = frases_filings.sample(frac=1, random_state=SEMILLA)
muestra_filings = barajadas.groupby("documento").head(FRASES_POR_FILING)
corpus = pd.concat([titulares_corpus, muestra_filings], ignore_index=True)
corpus["palabras"] = corpus["texto"].str.split().str.len()
corpus.to_parquet(CARPETA_CACHE / "corpus_preentrenamiento.parquet", index=False)

resumen_corpus = corpus.groupby("fuente").agg(frases=("texto", "size"), palabras=("palabras", "sum"), empresas=("ticker", "nunique"))
resumen_corpus["% de las palabras"] = (100 * resumen_corpus["palabras"] / resumen_corpus["palabras"].sum()).round(0)
print(f"Corpus: {len(corpus):,} frases · {corpus['palabras'].sum() / 1e6:.1f} M palabras · {corpus['ticker'].nunique()} empresas")
resumen_corpus

Corpus: 695,136 frases · 17.1 M palabras · 847 empresas


,frases,palabras,empresas,% de las palabras
fuente,,,,
filing_mda,319675,10868125,755,64.0
filing_riesgos,80925,2836979,744,17.0
titular,294536,3367079,638,20.0


**Conclusión:** el corpus junta el estilo corto de los titulares con el lenguaje largo y técnico de los informes, de cientos de empresas y casi cuatro años. La mitad de las frases de los filings estaban repetidas: las empresas copian el mismo texto legal trimestre tras trimestre. Con 17,1 M de palabras (64 % de MD&A, 20 % de titulares y 17 % de riesgos), cada preentrenamiento cabe en unos 25 minutos.

## Comprobaciones

In [24]:
# Noticias: solo nuestras empresas desde oct 2016, sin duplicados; las recientes mencionan siempre a su empresa
assert noticias["ticker"].isin(universo["ticker"]).all() and noticias["fecha"].min() >= pd.Timestamp("2016-10-01")
assert not noticias.duplicated(subset=["ticker", "titular"]).any()
assert not noticias_recientes.duplicated(subset=["ticker", "url"]).any()
assert all(menciona_empresa(t, k, claves[k]) for t, k in zip(noticias_recientes["titular"], noticias_recientes["ticker"]))

# La regla de mención, con casos escritos a mano
assert palabra_clave_empresa("AAL", "AMERICAN AIRLINES GROUP INC.") == "AMERICAN AIRLINES"
assert palabra_clave_empresa("TROW", "PRICE T ROWE GROUP INC") == "T ROWE"          # lista a mano
assert not universo["palabra_clave"].where(universo["palabra_clave"].str.len() >= 3).dropna().duplicated().any()  # ninguna compartida
assert menciona_empresa("T. Rowe Price Beats Estimates", "TROW", "T ROWE")
assert not menciona_empresa("Gold price today, October 21", "TROW", "T ROWE")
assert menciona_empresa("Bio-Techne Raises Guidance", "TECH", "BIO TECHNE")          # sin puntuación
assert not menciona_empresa("I tried a $99 plug", "PLUG", "PLUG POWER")              # ticker solo en mayúsculas
assert not menciona_empresa("PG&E Files for Bankruptcy", "PG", "PROCTER")            # PG dentro de PG&E
assert not menciona_empresa("A Big Day for U.S. Stocks", "A", "AGILENT")             # ticker de una letra suelto
assert menciona_empresa("Agilent Technologies (A) Q3 Earnings", "A", "")
assert menciona_empresa("FedEx (FDX) Stock Pops On Q3 Beat", "FDX", "FEDEX")
assert not menciona_empresa("Altria: Income And Growth Together", "TXN", "TEXAS INSTRUMENTS")
assert not menciona_empresa("Next steps for X-ray makers", "XE", "X")   # palabra clave demasiado corta
print("Noticias: OK")

Noticias: OK


In [25]:
# Filings: solo empresas de 10-K, casi todos procesados, ninguna sección empieza en una cita ni ocupa casi todo el documento
assert set(filings_texto["cik"]) <= set(empresas_sec.loc[empresas_sec["formulario_anual"] == "10-K", "cik"])
assert len(filings_texto) >= 0.98 * len(documentos)
assert (filings_texto["caracteres_riesgos"] <= MAXIMO_DEL_DOCUMENTO * filings_texto["caracteres_documento"]).all()
assert not filings_texto["riesgos"].str[:90].str.contains(r"^item\W*1a\W*risk\s+factors\W{0,3}[”\"]", case=False, regex=True).any()
assert ((filings_texto["ticker"] == "XOM") & (filings_texto["fecha_presentacion"] < "2020-07-01") & filings_texto["tiene_riesgos"]).any()

# La extracción, con un caso escrito a mano: índice, referencia, sección real y una cita con punto
caso = 'See Item 1A "Risk Factors". Contents Item 1A. Risk Factors 9 Item 1B. Unresolved 22 ' \
       'Item 1A. Risk Factors We face risks, as detailed in Item 2. Properties. More risks. Item 1B. Unresolved Staff Comments None.'
assert extraer_seccion(caso, "10-K", "riesgos") == "Item 1A. Risk Factors We face risks, as detailed in Item 2. Properties. More risks. "
assert extraer_seccion('in Item 1A. Risk Factors." This discussion is about results. Item 1B. Unresolved', "10-K", "riesgos") == ""

# Corpus: hasta junio de 2020, sin frases repetidas y sin ninguna noticia de NewsAPI; reglas de frases a mano
assert corpus["fecha"].max() <= pd.Timestamp(FECHA_LIMITE_CORPUS)
assert not corpus.duplicated(subset="texto").any()
assert not corpus["texto"].isin(noticias_recientes["titular"]).any()
assert partir_en_frases("Sales rose. Costs fell! Why? Because.") == ["Sales rose.", "Costs fell!", "Why?", "Because."]
assert not es_frase_util("See Part I, Item 1.") and not es_frase_util("Revenue $260,174 $265,595 $229,234 $215,639")
print("Filings y corpus: OK")

Filings y corpus: OK
